# 02 - Data Validation**Objective:** Validate wine records using Pydantic models and Pandera DataFrame schemas.**Tools:** Pydantic (row-level), Pandera (DataFrame-level)**Steps:**1. Define a Pydantic model for wine records2. Implement row-level validation3. Define a Pandera DataFrame schema4. Implement DataFrame-level validation5. Test on clean and corrupted data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathfrom pydantic import BaseModel, field_validatorimport pandera as pafrom pandera.typing import DataFrame, Seriesprint("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data from data/processed/clean_data.csv# We'll validate this data using both Pydantic and Pandera.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")# print(f"Data shape: {df.shape}")# print(f"Columns: {list(df.columns)}")

### Pydantic Model DefinitionPydantic validates data at the **record level** — each row is checked individually.Define a model with typed fields and add `@field_validator` decorators to enforcebusiness rules like valid class labels, positive measurements, etc.This catches row-level issues like a class value of -1, a negative Alcohol,or a proline of 10000 that might slip through type-only checks.

In [ ]:
# === Executed Example: Pydantic Row-Level Validation ===# This cell uses a small inline dataset (no CSV needed) to demonstrate# how Pydantic catches invalid rows at the record level.import pandas as pdfrom pydantic import BaseModel, field_validatordata = pd.DataFrame({    "id": [1, 2, 3, 4, 5],    "alcohol": [5.1, 7.2, -1.0, 99.9, 4.5],    "class_label": [0, 1, 2, 0, 3],})class Record(BaseModel):    id: int    alcohol: float    class_label: int    @field_validator("alcohol")    @classmethod    def alcohol_in_range(cls, v):        if not (0 < v <= 15):            raise ValueError(f"Alcohol out of range: {v}")        return v    @field_validator("class_label")    @classmethod    def valid_class(cls, v):        if v not in [0, 1, 2]:            raise ValueError(f"Class must be 0, 1, or 2, got {v}")        return vvalid = []for _, row in data.iterrows():    try:        Record(**row.to_dict())        valid.append(True)    except ValueError:        valid.append(False)print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [ ]:
# === Commented Template: Pydantic Row-Level Validation ===# Copy, paste, and adapt to your own dataset. Uncomment to use.# import pandas as pd# from pydantic import BaseModel, field_validator# data = pd.DataFrame({#     "field_1": [val1, val2, val3],#     "field_2": [val1, val2, val3],# })# class MyRecord(BaseModel):#     field_1: int#     field_2: float#     @field_validator("field_1")#     @classmethod#     def rule_name(cls, v):#         if not (lower <= v <= upper):#             raise ValueError(f"Constraint description: {v}")#         return v# valid = []# for _, row in data.iterrows():#     try:#         MyRecord(**row.to_dict())#         valid.append(True)#     except ValueError:#         valid.append(False)# print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [ ]:
# TODO: Define the Pydantic model for wine records# Each field needs a type hint. Add field_validators to enforce constraints.## Validation rules you might want:#   - class must be 0, 1, or 2#   - All measurements (alcohol, malic_acid, ash, total_phenols) must be positive#   - Chemical measurements should be within realistic ranges for wine cultivarsclass WineRecord(BaseModel):    """Pydantic model for a single wine record."""    # TODO: Add all fields with proper type hints    # alcohol: float    # malic_acid: float    # ash: float    # total_phenols: float    # class_label: int  (note: avoid naming field "class" — it shadows Python built-in)    # TODO: Add field validators for business rules    #    # @field_validator("class_label")    # def check_class(cls, v):    #     """Class must be 0 (cultivar_1), 1 (cultivar_2), or 2 (cultivar_3)."""    #     if v not in [0, 1, 2]:    #         raise ValueError(f"Class must be 0, 1, or 2, got {v}")    #     return v    pass

In [ ]:
# TODO: Implement a function that validates a DataFrame row-by-row using Pydantic# This function iterates over each row, tries to construct an WineRecord,# and collects whether the row passed or failed validation.def validate_with_pydantic(df):    """Validate each row of the DataFrame using the Pydantic model.    Returns a list of booleans where True means the row is valid.    Also prints a summary of how many rows passed vs failed.    """    # TODO: Loop through each row with df.iterrows()    # For each row, try:    #     WineRecord(**row.to_dict())    # If it succeeds, mark the row as valid (True), otherwise invalid (False)    #    # valid = []    # for _, row in df.iterrows():    #     try:    #         WineRecord(**row.to_dict())    #         valid.append(True)    #     except Exception:    #         valid.append(False)    #    # valid_count = sum(valid)    # invalid_count = len(valid) - valid_count    # print(f"Valid: {valid_count}, Invalid: {invalid_count}")    # return valid    pass# TODO: Test the Pydantic validation on the clean data# valid_flags = validate_with_pydantic(df)# print(f"Pass rate: {sum(valid_flags) / len(valid_flags):.1%}")

### Pandera Schema DefinitionPandera validates data at the **DataFrame level** — instead of checking one row at a time,it defines column-wide constraints (data types, value ranges, nullable rules, etc.).This is more efficient for bulk validation and catches issues like a columnthat unexpectedly contains strings instead of numbers.

In [ ]:
# === Executed Example: Pandera DataFrame-Level Validation ===# This cell uses the same inline dataset to demonstrate how Pandera# validates all rows at once, collecting all failures.import pandas as pdimport pandera as padata = pd.DataFrame({    "id": [1, 2, 3, 4, 5],    "alcohol": [5.1, 7.2, -1.0, 99.9, 4.5],    "class": [0, 1, 2, 0, 3],})Schema = pa.DataFrameSchema({    "id": pa.Column(pa.Int, nullable=False),    "alcohol": pa.Column(pa.Float, checks=[pa.Check.greater_than(0), pa.Check.le(15)]),    "class": pa.Column(pa.Int, checks=pa.Check.isin([0, 1, 2])),})try:    Schema.validate(data, lazy=True)    print("Pandera: all rows passed")except pa.errors.SchemaErrors as e:    print(f"Pandera: {len(e.failure_cases)} failures found")    print(e.failure_cases[["index", "column", "check", "failure_case"]].head())

In [ ]:
# === Commented Template: Pandera DataFrame-Level Validation ===# Copy, paste, and adapt to your own dataset. Uncomment to use.# import pandas as pd# import pandera as pa# data = pd.DataFrame({#     "field_1": [val1, val2, val3],#     "field_2": [val1, val2, val3],# })# MySchema = pa.DataFrameSchema({#     "field_1": pa.Column(pa.Int, nullable=False),#     "field_2": pa.Column(pa.Float, checks=[pa.Check.ge(min_val), pa.Check.le(max_val)]),# })# try:#     MySchema.validate(data, lazy=True)#     print("Pandera: all rows valid")# except pa.errors.SchemaErrors as e:#     print(f"Pandera: {len(e.failure_cases)} failure(s)")

In [ ]:
# TODO: Define a Pandera DataFrameSchema for wine data# Each column gets a pa.Column with a dtype and optional Check constraints.## For example:#   "class": pa.Column(pa.Int, checks=pa.Check.isin([0, 1, 2]))# ensures the class column only contains valid wine cultivars labels.## Measurement columns should be within realistic ranges:#   "alcohol": pa.Column(pa.Float, checks=pa.Check.greater_than(0))# WineSchema = pa.DataFrameSchema({#     "class": pa.Column(pa.Int, checks=pa.Check.isin([0, 1, 2])),#     "alcohol": pa.Column(pa.Float, checks=pa.Check.greater_than(0)),#     "malic_acid": pa.Column(pa.Float, checks=pa.Check.greater_than(0)),#     "ash": pa.Column(pa.Float, checks=pa.Check.greater_than(0)),#     "total_phenols": pa.Column(pa.Float, checks=pa.Check.greater_than(0)),# })

In [ ]:
# TODO: Implement a function that validates a DataFrame using the Pandera schema# Call WineSchema.validate(df) which raises an exception if validation fails.# Wrap it in a try/except to handle the error gracefully.def validate_with_pandera(df):    """Validate the entire DataFrame using the Pandera schema."""    # TODO: Call WineSchema.validate(df) inside a try block    # If it succeeds, print "Pandera validation passed"    # If it raises an exception, print the error message    pass# TODO: Test the Pandera validation on the clean data# validate_with_pandera(df)

In [ ]:
# Exercise: Test validation on corrupted data# The data_injection module can create corrupted variants of the clean data.# Run both Pydantic and Pandera validation on corrupted data and observe# what kinds of issues each validation method catches.## TODO: Load corrupted data and run both validators# from data_injection.injector import generate_corrupted_dataset# from data_injection.config import CORRUPTION_PRESETS# df_clean = pd.read_csv("../data/processed/clean_data.csv")# df_corrupted = generate_corrupted_dataset(df_clean, CORRUPTION_PRESETS["missing_heavy"])## valid_flags = validate_with_pydantic(df_corrupted)# print(f"Corrupted data pass rate: {sum(valid_flags) / len(valid_flags):.1%}")## validate_with_pandera(df_corrupted)

### Exercises1. **Add more validators**: Add a `@field_validator` for `ash` that rejects values below 0 or above 10 (a realistic range for Wine).2. **Compare error messages**: Run the same corrupted data through both Pydantic and Pandera — which one gives more informative error messages?3. **Performance test**: Time both validators on the full dataset (use `%%timeit` in a cell) — is Pandera noticeably faster for bulk validation?4. **Custom error handler**: Modify `validate_with_pydantic()` to collect and return the actual error messages (not just True/False) so students can see why a row failed.5. **Schema evolution**: What happens if a new column is added to the CSV? Will Pandera reject it or pass it through? Check the `strict` parameter in `pa.DataFrameSchema`.